# San Diego County Rental Property Map

Builds the definitive inventory of every rental property in San Diego County
using SANDAG parcel data (1.09M parcels). Answers:

> There are **X** rental properties in San Diego County containing **Y** total units.

Distinguishes single-family rentals, duplexes, small multifamily,
apartment buildings, and commercial/other rental properties.

## 1. Load Parcel Data

In [ ]:
from pathlib import Path

import folium
import polars as pl

DATA_DIR = Path("data")
parcels = pl.read_parquet(DATA_DIR / "parcels.parquet")
print(f"Total parcels: {parcels.shape[0]:,}")
print(f"Columns: {parcels.columns}")

## 2. Land Use Classification

Map assessor land use codes (`ASR_LANDUS`) to human-readable categories
and a broad property class for analysis.

In [ ]:
LANDUSE_LABELS = {
    0: "Unclassified",
    6: "Information Parcel",
    7: "Timeshare",
    9: "Manufactured Home in Park",
    10: "Vacant Residential",
    11: "Single Family Residential",
    12: "Duplex",
    13: "Multifamily 2-4 Units",
    14: "Multifamily 5-15 Units",
    15: "Multifamily 16-60 Units",
    16: "Multifamily 61+ Units",
    17: "Condo / Other Residential",
    18: "Co-op",
    19: "Special / Small Parcel",
    20: "Vacant Commercial",
    21: "Commercial Office/Retail 1-3 Stories",
    22: "Commercial Office 4+ Stories",
    23: "Regional Shopping Center",
    24: "Community Shopping Center",
    25: "Neighborhood Shopping Center",
    26: "Hotel / Motel",
    27: "Service Station",
    28: "Medical / Dental Office",
    29: "Rest Home / Convalescent",
    30: "Office Condo",
    31: "Parking / Used Car Lot",
    32: "Trailer Park",
    33: "Theater",
    34: "Bowling Alley",
    35: "Restaurant / Nightclub",
    36: "Car Wash",
    37: "Grocery / Drug Chain",
    38: "Auto Sales / Service",
    39: "Bank / Radio / Misc Commercial",
    40: "Vacant Industrial",
    41: "Light Manufacturing",
    42: "Heavy Manufacturing",
    43: "Warehouse / Distribution",
    44: "Bulk Chemical / Oil Refinery",
    45: "Mining / Extractive",
    46: "Auto Repair Garage",
    47: "Industrial Condo",
    49: "Misc Industrial",
    50: "Irrigated Farm Vacant",
    51: "Citrus",
    52: "Avocado",
    53: "Vines",
    54: "Misc Trees",
    55: "Livestock",
    56: "Poultry",
    57: "Irrigated Crops",
    58: "Growing Houses",
    59: "Misc Agricultural",
    61: "Rural Land 1-10 Acres",
    62: "Rural Land 11-40 Acres",
    63: "Rural Land 41-160 Acres",
    64: "Rural Land 161-360 Acres",
    65: "Rural Land 361+ Acres",
    70: "Institutional Vacant",
    71: "Church",
    72: "Church Related",
    73: "Cemetery",
    74: "Mausoleum",
    75: "Mortuary",
    76: "Public Building",
    77: "Hospital",
    78: "Private School",
    79: "Misc Institutional",
    80: "Vacant Recreational",
    81: "Meeting Hall / Gym",
    82: "Golf Course",
    83: "Marina / Docks",
    84: "Recreational Camp",
    85: "Non-Taxable",
    86: "Open Space",
    87: "Ag Preserve (No Contract)",
    88: "Ag Preserve (Contract)",
    89: "Misc / Special",
    90: "Vacant Govt Property",
    91: "Improved Govt Property",
}

RENTAL_CLASSES = {
    9: "Manufactured Home",
    11: "Single Family",
    12: "Duplex",
    13: "Small Multifamily (2-4)",
    14: "Apartment (5-15)",
    15: "Apartment (16-60)",
    16: "Apartment (61+)",
    17: "Condo",
    18: "Co-op",
    26: "Hotel / Motel",
    29: "Rest Home / Convalescent",
    32: "Trailer Park",
}

label_df = pl.DataFrame({
    "ASR_LANDUS": list(LANDUSE_LABELS.keys()),
    "landuse_label": list(LANDUSE_LABELS.values()),
})

class_df = pl.DataFrame({
    "ASR_LANDUS": list(RENTAL_CLASSES.keys()),
    "rental_class": list(RENTAL_CLASSES.values()),
})

parcels = (
    parcels
    .join(label_df, on="ASR_LANDUS", how="left")
    .join(class_df, on="ASR_LANDUS", how="left")
)

print("Parcel counts by land use (top 20):")
(
    parcels
    .group_by("ASR_LANDUS", "landuse_label")
    .agg(pl.len().alias("parcels"), pl.col("UNITQTY").sum().alias("total_units"))
    .sort("parcels", descending=True)
    .head(20)
)

## 3. Identify Rental Properties

A parcel is classified as a rental property when:
- Its land use code is a residential type (SFR, duplex, multifamily, condo, manufactured home)
- **AND** it is explicitly not owner-occupied (`OWNEROCC` is a non-null, non-Y value)
- Parcels with null `OWNEROCC` are reported separately as unknown occupancy

Rest homes and trailer parks are shown separately as commercial rentals.

In [ ]:
RESIDENTIAL_CODES = {9, 11, 12, 13, 14, 15, 16, 17, 18}
COMMERCIAL_RENTAL_CODES = {29, 32}

residential = parcels.filter(
    pl.col("ASR_LANDUS").is_in(RESIDENTIAL_CODES)
)

rentals = residential.filter(
    pl.col("OWNEROCC").is_not_null() & (pl.col("OWNEROCC") != "Y")
)
owner_occupied = residential.filter(pl.col("OWNEROCC") == "Y")
unknown_occ = residential.filter(pl.col("OWNEROCC").is_null())
commercial_rentals = parcels.filter(pl.col("ASR_LANDUS").is_in(COMMERCIAL_RENTAL_CODES))

print("SAN DIEGO COUNTY (all jurisdictions)")
print(f"  All residential parcels:       {residential.shape[0]:>10,}")
print(f"    Owner-occupied:              {owner_occupied.shape[0]:>10,}")
print(f"    Rental (confirmed non-owner):{rentals.shape[0]:>10,}")
print(f"    Unknown (null OWNEROCC):     {unknown_occ.shape[0]:>10,}")
print(f"    Commercial rental:           {commercial_rentals.shape[0]:>10,}")
print()

rental_units = rentals["UNITQTY"].sum()
commercial_units = commercial_rentals["UNITQTY"].sum()

print(f"  Residential rental units:  {rental_units:>10,}")
print(f"  Commercial rental units:   {commercial_units:>10,}")
print(f"  Total rental units:        {rental_units + commercial_units:>10,}")

# City of San Diego only (SITUS_JURI='SD') for STRO/TOT comparison later
city_parcels = parcels.filter(pl.col("SITUS_JURI") == "SD")
city_residential = city_parcels.filter(pl.col("ASR_LANDUS").is_in(RESIDENTIAL_CODES))
city_rentals = city_residential.filter(
    pl.col("OWNEROCC").is_not_null() & (pl.col("OWNEROCC") != "Y")
)
city_owner = city_residential.filter(pl.col("OWNEROCC") == "Y")
city_unknown = city_residential.filter(pl.col("OWNEROCC").is_null())
city_commercial = city_parcels.filter(pl.col("ASR_LANDUS").is_in(COMMERCIAL_RENTAL_CODES))

print()
print("CITY OF SAN DIEGO (SITUS_JURI='SD')")
print(f"  Residential parcels:           {city_residential.shape[0]:>10,}")
print(f"  Residential units:             {city_residential['UNITQTY'].sum():>10,}")
print(f"    Owner-occupied:              {city_owner.shape[0]:>10,}")
print(f"    Rental (confirmed non-owner):{city_rentals.shape[0]:>10,}")
print(f"    Unknown (null OWNEROCC):     {city_unknown.shape[0]:>10,}")
print(f"  Rental units:                  {city_rentals['UNITQTY'].sum():>10,}")

## 4. Breakdown by Property Type

In [ ]:
all_rentals = pl.concat([
    rentals,
    commercial_rentals,
], how="diagonal_relaxed")

breakdown = (
    all_rentals
    .group_by("rental_class")
    .agg(
        pl.len().alias("properties"),
        pl.col("UNITQTY").sum().alias("total_units"),
        pl.col("UNITQTY").mean().alias("avg_units"),
        pl.col("UNITQTY").max().alias("max_units"),
        pl.col("ASR_TOTAL").mean().alias("avg_assessed_value"),
    )
    .sort("total_units", descending=True)
)

print("Rental properties by type:")
breakdown

## 5. Geographic Distribution by Community

In [ ]:
community_stats = (
    all_rentals
    .filter(pl.col("SITUS_COMM").is_not_null() & (pl.col("SITUS_COMM") != ""))
    .group_by("SITUS_COMM")
    .agg(
        pl.len().alias("properties"),
        pl.col("UNITQTY").sum().alias("total_units"),
        pl.col("ASR_TOTAL").mean().alias("avg_assessed_value"),
    )
    .sort("total_units", descending=True)
)

print(f"Communities with rental properties: {community_stats.shape[0]}")
print()
print("Top 25 communities by rental unit count:")
community_stats.head(25)

## 6. Apartment Complexes (5+ Units)

Largest apartment buildings in the county.

In [ ]:
apartments = all_rentals.filter(pl.col("UNITQTY") >= 5)

print(f"Apartment complexes (5+ units): {apartments.shape[0]:,}")
print(f"Total apartment units: {apartments['UNITQTY'].sum():,}")
print()

print("Top 25 largest complexes:")
(
    apartments
    .select(
        "APN",
        pl.concat_str(
            [pl.col("SITUS_ADDR").cast(pl.String), pl.col("SITUS_PRE_"), pl.col("SITUS_STRE"), pl.col("SITUS_SUFF")],
            separator=" ",
        ).str.replace_all(r"\s+", " ").str.strip_chars().alias("address"),
        "SITUS_COMM",
        "SITUS_ZIP",
        "UNITQTY",
        "TOTAL_LVG_",
        "ASR_TOTAL",
        "rental_class",
    )
    .sort("UNITQTY", descending=True)
    .head(25)
)

## 7. Single Family Rental Concentration

Zip codes with the highest share of non-owner-occupied single family homes.

In [ ]:
sfr = residential.filter(pl.col("ASR_LANDUS") == 11)

sfr_by_zip = (
    sfr
    .filter(pl.col("SITUS_ZIP").is_not_null() & (pl.col("SITUS_ZIP") != ""))
    .group_by("SITUS_ZIP")
    .agg(
        pl.len().alias("total_sfr"),
        (pl.col("OWNEROCC") != "Y").sum().alias("rental_sfr"),
    )
    .with_columns(
        (pl.col("rental_sfr") / pl.col("total_sfr") * 100).round(1).alias("rental_pct"),
    )
    .filter(pl.col("total_sfr") >= 100)
    .sort("rental_pct", descending=True)
)

print("Zip codes with highest SFR rental rate (min 100 homes):")
sfr_by_zip.head(20)

## 8. Summary

In [ ]:
total_properties = all_rentals.shape[0]
total_units = all_rentals["UNITQTY"].sum()

print("=" * 60)
print(f"SAN DIEGO COUNTY RENTAL PROPERTY INVENTORY")
print("=" * 60)
print(f"Total rental properties: {total_properties:>12,}")
print(f"Total rental units:     {total_units:>12,}")
print("-" * 60)

for row in breakdown.sort("total_units", descending=True).to_dicts():
    cls = row["rental_class"] or "Unknown"
    print(f"  {cls:<30} {row['properties']:>8,} properties  {row['total_units']:>10,} units")

print("-" * 60)
print(f"  {'TOTAL':<30} {total_properties:>8,} properties  {total_units:>10,} units")
print("=" * 60)

## 9. Map: Apartment Complex Density

Heatmap of apartment complexes (5+ units) across San Diego County.
Coordinates are in CA State Plane Zone 6 (EPSG:2230, feet) and need
conversion to WGS84 lat/lng.

In [ ]:
import math


def state_plane_to_wgs84(x_ft: float, y_ft: float) -> tuple[float, float]:
    """Approximate conversion from CA State Plane Zone 6 (NAD83, feet) to WGS84.

    Uses a local affine approximation calibrated to San Diego County center.
    Accuracy is within ~50m across the county, sufficient for mapping.
    """
    # Reference point: roughly center of SD county
    x0, y0 = 6_280_000.0, 1_920_000.0
    lat0, lng0 = 32.83, -117.08

    # Scale factors (feet to degrees) at this latitude
    ft_per_deg_lat = 364_567.0
    ft_per_deg_lng = 308_031.0

    lat = lat0 + (y_ft - y0) / ft_per_deg_lat
    lng = lng0 + (x_ft - x0) / ft_per_deg_lng
    return lat, lng


apt_with_coords = (
    apartments
    .filter(
        pl.col("X_COORD").is_not_null()
        & pl.col("Y_COORD").is_not_null()
        & (pl.col("X_COORD") > 0)
        & (pl.col("Y_COORD") > 0)
    )
)

coords = [
    state_plane_to_wgs84(row["X_COORD"], row["Y_COORD"])
    for row in apt_with_coords.select("X_COORD", "Y_COORD").to_dicts()
]

apt_mapped = apt_with_coords.with_columns(
    pl.Series("lat", [c[0] for c in coords]),
    pl.Series("lng", [c[1] for c in coords]),
)

print(f"Apartment complexes with coordinates: {apt_mapped.shape[0]:,}")

In [ ]:
from folium.plugins import HeatMap

m = folium.Map(location=[32.83, -117.08], zoom_start=10, tiles="CartoDB positron")

heat_data = [
    [row["lat"], row["lng"], min(row["UNITQTY"], 300)]
    for row in apt_mapped.select("lat", "lng", "UNITQTY").to_dicts()
]

HeatMap(
    heat_data,
    radius=12,
    blur=15,
    max_zoom=13,
    min_opacity=0.3,
).add_to(m)

print(f"Heatmap: {len(heat_data):,} apartment complexes, weighted by unit count")
m

## 10. Cross-Reference: Short-Term Rental (STRO) Licenses

STRO licenses and TOT certificates are issued by the **City of San Diego**,
so this section filters parcels to `SITUS_JURI='SD'` for an apples-to-apples
comparison. The three-pass address matching handles:
1. Exact match on street number + street name + zip
2. Fallback without zip (boundary zip mismatches)
3. Nearest address number on the same street within 20 house numbers

In [ ]:
from datetime import datetime

RENTAL_DATA_DIR = Path("..") / "rental_properties" / "data"

stro = pl.read_csv(RENTAL_DATA_DIR / "stro_licenses_datasd.csv", infer_schema_length=10000)
stro = stro.with_columns(
    pl.col("latitude").cast(pl.Float64, strict=False),
    pl.col("longitude").cast(pl.Float64, strict=False),
    pl.col("date_expiration").str.to_datetime("%Y-%m-%d", strict=False),
)
stro_active = stro.filter(pl.col("date_expiration") >= datetime.now())

tot = pl.read_csv(RENTAL_DATA_DIR / "tot_establishments_datasd.csv", infer_schema_length=10000)

airbnb = pl.read_csv(
    Path("..") / "airbnb" / "data" / "visualisations_listings.csv",
    infer_schema_length=10000,
)

print(f"STRO licenses loaded:  {stro.shape[0]:,}  (active: {stro_active.shape[0]:,})")
print(f"TOT certificates loaded: {tot.shape[0]:,}")
print(f"Airbnb listings loaded:  {airbnb.shape[0]:,}  (Inside Airbnb, metro-wide)")

In [ ]:
def normalize_street(s: pl.Expr) -> pl.Expr:
    """Normalize a street name for address matching."""
    return (
        s
        .str.to_uppercase()
        .str.strip_chars()
        .str.replace_all(r"\s+", " ")
        # Strip leading zeros from numbered streets: 03RD -> 3RD, 06TH -> 6TH
        .str.replace(r"^0+(\d)", "$1")
        # Normalize abbreviations
        .str.replace_all(r"\bMT\.\s*", "MOUNT ")
        .str.replace_all(r"\bST\.\s*", "SAINT ")
        # Remove stray punctuation
        .str.replace_all(r"[%#]", "")
        .str.replace_all(r"\s+", " ")
        .str.strip_chars()
    )


PARCEL_AGG_EXPRS = [
    pl.col("APN").first(),
    pl.col("ASR_LANDUS").first(),
    pl.col("UNITQTY").sum().alias("total_units_at_address"),
    pl.col("ASR_TOTAL").sum().alias("total_assessed_value"),
    pl.col("OWNEROCC").first(),
    pl.col("SITUS_COMM").first(),
    pl.col("rental_class").first(),
    pl.col("landuse_label").first(),
    pl.len().alias("parcel_count"),
]

parcels_norm = parcels.with_columns(
    pl.col("SITUS_ADDR").cast(pl.String).str.strip_chars().alias("join_num"),
    normalize_street(pl.col("SITUS_STRE")).alias("join_street"),
    pl.col("SITUS_ZIP").str.slice(0, 5).alias("join_zip"),
).filter(
    (pl.col("join_num") != "0") & (pl.col("join_num") != "")
    & pl.col("join_street").is_not_null() & (pl.col("join_street") != "")
    & pl.col("join_zip").is_not_null() & (pl.col("join_zip") != "")
)

# Pass 1 lookup: number + street + zip
parcel_addr = parcels_norm.group_by("join_num", "join_street", "join_zip").agg(*PARCEL_AGG_EXPRS)

# Pass 2 lookup: number + street (no zip)
parcel_addr_nozip = parcels_norm.group_by("join_num", "join_street").agg(*PARCEL_AGG_EXPRS)

# Pass 3 lookup: nearest address number on same street + zip.
# For each (street, zip) keep a list of known address numbers so we can
# find the closest match for STRO addresses that fall between parcels.
parcel_street_zip = (
    parcels_norm
    .with_columns(pl.col("join_num").cast(pl.Int64, strict=False).alias("num_int"))
    .filter(pl.col("num_int").is_not_null() & (pl.col("num_int") > 0))
    .group_by("join_street", "join_zip", "num_int")
    .agg(*PARCEL_AGG_EXPRS)
)

stro_norm = stro.with_columns(
    pl.col("street_number").cast(pl.String).str.strip_chars().alias("join_num"),
    normalize_street(pl.col("street_name")).alias("join_street"),
    pl.col("zip").cast(pl.String).str.slice(0, 5).alias("join_zip"),
)

# --- Pass 1: exact match on number + street + zip ---
pass1 = stro_norm.join(parcel_addr, on=["join_num", "join_street", "join_zip"], how="left")
matched_p1 = pass1.filter(pl.col("APN").is_not_null())
unmatched_p1 = pass1.filter(pl.col("APN").is_null()).select(stro_norm.columns)

# --- Pass 2: number + street, drop zip ---
pass2 = unmatched_p1.join(parcel_addr_nozip, on=["join_num", "join_street"], how="left")
matched_p2 = pass2.filter(pl.col("APN").is_not_null())
unmatched_p2 = pass2.filter(pl.col("APN").is_null()).select(stro_norm.columns)

# --- Pass 3: nearest address number on same street + zip ---
# Join unmatched STRO to all parcels on same street+zip, pick closest number.
unmatched_with_int = unmatched_p2.with_columns(
    pl.col("join_num").cast(pl.Int64, strict=False).alias("stro_num_int")
).filter(pl.col("stro_num_int").is_not_null())

pass3 = (
    unmatched_with_int
    .join(parcel_street_zip, on=["join_street", "join_zip"], how="inner")
    .with_columns(
        (pl.col("stro_num_int") - pl.col("num_int")).abs().alias("addr_dist")
    )
    .filter(pl.col("addr_dist") <= 20)
    .sort("addr_dist")
    .group_by("license_id")
    .first()
)
matched_p3 = pass3.filter(pl.col("APN").is_not_null())

stro_parcels = pl.concat([matched_p1, matched_p2, matched_p3], how="diagonal_relaxed")
matched = stro_parcels.filter(pl.col("APN").is_not_null())
total_unmatched = stro.shape[0] - matched.shape[0]

print(f"STRO licenses:       {stro.shape[0]:>6,}")
print(f"Pass 1 (num+street+zip):           {matched_p1.shape[0]:>6,}")
print(f"Pass 2 (num+street, no zip):       {matched_p2.shape[0]:>6,}")
print(f"Pass 3 (nearest addr, +/-20):      {matched_p3.shape[0]:>6,}")
print(f"Total matched:       {matched.shape[0]:>6,}  ({matched.shape[0] / stro.shape[0] * 100:.1f}%)")
print(f"Unmatched:           {total_unmatched:>6,}  ({total_unmatched / stro.shape[0] * 100:.1f}%)")

In [ ]:
stro_by_type = (
    matched
    .group_by("landuse_label")
    .agg(
        pl.len().alias("stro_licenses"),
        pl.col("total_assessed_value").mean().round(0).alias("avg_assessed_value"),
    )
    .sort("stro_licenses", descending=True)
)

print("STRO licenses by parcel land use type:")
stro_by_type

In [ ]:
stro_by_tier = (
    matched
    .group_by("tier", "landuse_label")
    .agg(pl.len().alias("count"))
    .sort("tier", "count", descending=[False, True])
)

print("STRO licenses by tier and property type:")
stro_by_tier

In [ ]:
owner_occ_count = matched.filter(pl.col("OWNEROCC") == "Y").shape[0]
non_owner_count = matched.filter(pl.col("OWNEROCC") != "Y").shape[0]

print("STRO licenses: owner-occupied vs investor-owned")
print(f"  Owner-occupied:     {owner_occ_count:>5,}  ({owner_occ_count / matched.shape[0] * 100:.1f}%)")
print(f"  Non-owner-occupied: {non_owner_count:>5,}  ({non_owner_count / matched.shape[0] * 100:.1f}%)")
print()

stro_community = (
    matched
    .group_by("SITUS_COMM")
    .agg(
        pl.len().alias("stro_licenses"),
        (pl.col("OWNEROCC") == "Y").sum().alias("owner_occupied"),
        (pl.col("OWNEROCC") != "Y").sum().alias("investor_owned"),
    )
    .sort("stro_licenses", descending=True)
)

print("Top 15 communities by STRO license count:")
stro_community.head(15)

### STRO as a Share of the City Rental Inventory

How much of the City of San Diego's rental housing stock is being used for
short-term rentals? Uses city-scoped parcel data (`SITUS_JURI='SD'`).

In [ ]:
city_all_rentals = pl.concat([city_rentals, city_commercial], how="diagonal_relaxed")

city_rental_by_comm = (
    city_all_rentals
    .filter(pl.col("SITUS_COMM").is_not_null() & (pl.col("SITUS_COMM") != ""))
    .group_by("SITUS_COMM")
    .agg(
        pl.col("UNITQTY").sum().alias("rental_units"),
    )
)

stro_vs_rental = (
    stro_community
    .join(city_rental_by_comm, on="SITUS_COMM", how="left")
    .with_columns(
        (pl.col("stro_licenses") / pl.col("rental_units") * 100)
        .round(2)
        .alias("stro_pct_of_rental_units"),
    )
    .filter(pl.col("rental_units") >= 100)
    .sort("stro_pct_of_rental_units", descending=True)
)

print("Communities where STROs consume the largest share of rental units:")
stro_vs_rental.head(20)

## 11. Cross-Reference: TOT Certificates (Excluding Hotels)

TOT (Transient Occupancy Tax) certificates cover properties collecting
lodging tax. The TOT dataset used here only contains vacation-type
certificates (whole-home and home-share); hotel and motel certificates
are not included in the source data.

In [ ]:
tot_parsed = tot.with_columns(
    pl.col("property_address").str.to_uppercase().str.strip_chars().alias("raw_addr"),
    pl.col("property_zip").cast(pl.String).str.slice(0, 5).alias("join_zip"),
).with_columns(
    pl.col("raw_addr").str.extract(r"^(\d+)\s+", 1).alias("join_num"),
    normalize_street(
        pl.col("raw_addr")
        .str.replace(r"^\d+\s+", "")
        .str.replace(r"\s+UNIT\s+.*$", "")
        .str.replace(r"\s+#\s*.*$", "")
        .str.replace(
            r"\s+(AVE?|BLVD?|ST|CT|DR|PL|WAY|RD|LN|CIR|TER|PKWY|HWY|CV|WY|WALK)$",
            "",
        )
    ).alias("join_street"),
)

tot_joined = tot_parsed.join(
    parcel_addr, on=["join_num", "join_street", "join_zip"], how="left",
)
tot_matched = tot_joined.filter(pl.col("APN").is_not_null())
tot_hotels = tot_matched.filter(pl.col("ASR_LANDUS") == 26)
tot_residential = tot_matched.filter(pl.col("ASR_LANDUS") != 26)

print(f"TOT certificates:          {tot.shape[0]:>8,}")
print(f"Matched to parcel:         {tot_matched.shape[0]:>8,}  ({tot_matched.shape[0] / tot.shape[0] * 100:.1f}%)")
print(f"  Hotels/motels:           {tot_hotels.shape[0]:>8,}")
print(f"  Residential (non-hotel): {tot_residential.shape[0]:>8,}")
print(f"Unmatched:                 {tot.shape[0] - tot_matched.shape[0]:>8,}")
print()

print("Matched TOT by land use:")
(
    tot_matched
    .group_by("landuse_label")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
    .head(15)
)

## 12. City of San Diego: STR vs Housing Stock Summary

In [ ]:
city_res_units = city_residential["UNITQTY"].sum()
city_rental_units = city_rentals["UNITQTY"].sum()
n_stro = stro_active.shape[0]
n_tot_res = tot_residential.shape[0]
n_tot_hotels = tot_hotels.shape[0]
n_airbnb = airbnb.shape[0]

print("=" * 65)
print("CITY OF SAN DIEGO: SHORT-TERM RENTALS vs HOUSING STOCK")
print("=" * 65)
print()
print(f"HOUSING STOCK")
print(f"  Residential units:             {city_res_units:>10,}")
print(f"  Rental units (non-owner-occ):  {city_rental_units:>10,}")
print()
print(f"SHORT-TERM RENTAL LICENSES")
print(f"  Active STRO licenses:          {n_stro:>10,}")
print(f"  TOT certificates (total):      {tot.shape[0]:>10,}")
print(f"    Hotels/motels (excluded):     {n_tot_hotels:>10,}")
print(f"    Residential TOT:             {n_tot_res:>10,}")
print()
print(f"AIRBNB (Inside Airbnb dataset):  {n_airbnb:>10,}")
print()
print("  CAVEAT: The Inside Airbnb dataset covers the entire San Diego")
print("  metro area (city + surrounding jurisdictions), not just the")
print(f"  city of San Diego. The {n_airbnb:,} Airbnb listings cannot be")
print(f"  directly compared to the {n_stro:,} city-only STRO licenses")
print("  without first filtering to city boundaries.")
print()
print("-" * 65)
print(f"STRO as % of housing stock:")
print(f"  vs all residential units:  {n_stro:,} / {city_res_units:,}  = {n_stro / city_res_units * 100:.2f}%")
print(f"  vs rental units only:      {n_stro:,} / {city_rental_units:,}  = {n_stro / city_rental_units * 100:.2f}%")
print()
print(f"TOT (excl. hotels) as % of housing stock:")
print(f"  vs all residential units:  {n_tot_res:,} / {city_res_units:,}  = {n_tot_res / city_res_units * 100:.2f}%")
print(f"  vs rental units only:      {n_tot_res:,} / {city_rental_units:,}  = {n_tot_res / city_rental_units * 100:.2f}%")
print()

city_bd = (
    city_rentals
    .join(class_df, on="ASR_LANDUS", how="left")
    .group_by("rental_class")
    .agg(pl.len().alias("properties"), pl.col("UNITQTY").sum().alias("units"))
    .sort("units", descending=True)
)

print("City rental inventory by type:")
for row in city_bd.to_dicts():
    print(f"  {row['rental_class']:<25} {row['properties']:>8,} properties  {row['units']:>9,} units")
print("=" * 65)